# Pretrained YOLOv8n Baseline
Author: Ahmed El Basyouni

Run pretrained YOLOv8n on the same new images. Its generic pretrained classes do not include a dedicated concrete-crack class. This illustrates the need for task-specific training; it is not a crack-detector accuracy benchmark.

In [ ]:
%pip install -q ultralytics==8.3.40 PyYAML==6.0.2


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, urllib.request, zipfile, json, time, random
import yaml, torch, ultralytics, pandas as pd
from ultralytics import YOLO
from IPython.display import display, Image
STARTED_AT = datetime.now(timezone.utc).isoformat()
START_TIME = time.perf_counter()
print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
RESULTS = Path('/content/results'); RESULTS.mkdir(exist_ok=True)
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()
def download_verified(url, path, expected_hash):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists() or sha256_file(path) != expected_hash:
        urllib.request.urlretrieve(url, path)
    actual = sha256_file(path)
    if actual != expected_hash:
        path.unlink(missing_ok=True)
        raise ValueError(f'Checksum mismatch: {actual}')
    return path


In [ ]:
model = YOLO('yolov8n.pt')
EVIDENCE = RESULTS/'baseline'


In [ ]:
NEW_IMAGES_DIR = Path('/content/new_images'); NEW_IMAGES_DIR.mkdir(exist_ok=True)
NEW_OUT = EVIDENCE/'new_images'; NEW_OUT.mkdir(parents=True, exist_ok=True)
summary = []
for i in range(1,6):
    name = f'new_{i:02d}.png'; path = NEW_IMAGES_DIR/name
    urllib.request.urlretrieve(
        f'https://raw.githubusercontent.com/aalbassiony-png/concrete-crack-detection/main/new_images/{name}', path)
    result = model.predict(str(path), imgsz=640, conf=0.25, iou=0.70,
                           device=DEVICE, verbose=False)[0]
    result.save(filename=str(NEW_OUT/name))
    summary.append({'image':name,'detections':len(result.boxes),
                    'confidences':result.boxes.conf.cpu().tolist()})
    display(Image(filename=str(NEW_OUT/name)))
display(pd.DataFrame(summary))
pd.DataFrame(summary).to_csv(NEW_OUT/'prediction_summary.csv',index=False)


## Interpretation
Inspect any generic-class detections. Do not interpret absence of generic detections as absence of cracks. No crack-specific baseline metric is claimed.